In [16]:
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: Load the Dataset
# ==========================================
# Read the CSV file into a pandas DataFrame

file_path = r"/Users/work/workspace/cardiac_failure/labs.csv"
data = pd.read_csv(file_path)

# Use 'data.shape' instead of 'df.shape'
print(f"1. Initial dataset loaded successfully. Shape: {data.shape}")

1. Initial dataset loaded successfully. Shape: (2008, 107)


In [1]:
print(df.head())

NameError: name 'df' is not defined

In [3]:
data = pd.read_csv(r"/Users/work/workspace/cardiac_failure/labs.csv")

In [5]:
print(data.head())
print(data.shape)
print(data.columns)
print(data.tail())

   inpatient_number  body_temperature  pulse  respiration  \
0            857781              36.7     87           19   
1            743087              36.8     95           18   
2            866418              36.5     98           18   
3            775928              36.0     73           19   
4            810128              35.0     88           19   

   systolic_blood_pressure  diastolic_blood_pressure  map_value  fio2  \
0                      102                        64  76.666667    33   
1                      150                        70  96.666667    33   
2                      102                        67  78.666667    33   
3                      110                        74  86.000000    33   
4                      134                        62  86.000000    33   

   creatinine_enzymatic_method   urea  ...  measured_residual_base  \
0                        108.3  12.55  ...                    -2.1   
1                         62.0   4.29  ...            

In [7]:
# 1. Count the total number of duplicate rows
duplicate_count = data.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")

Total duplicate rows found: 0


In [8]:
# Find columns where all values are null
empty_cols = [col for col in data.columns if data[col].isnull().all()]

print(f"Total columns in dataset: {len(data.columns)}")
if empty_cols:
    print(f"100% empty columns found: {empty_cols}")
else:
    print("No 100% empty columns found.")

Total columns in dataset: 107
100% empty columns found: ['cholinesterase']


In [10]:
#STEP 3: Drop 100% Empty Columns
# ==========================================
# Identify columns where every single row is missing (NaN)
empty_cols = [col for col in data.columns if data[col].isnull().all()]
data = data.drop(columns=empty_cols)
print(f"3. Dropped completely empty columns: {empty_cols}")

3. Dropped completely empty columns: ['cholinesterase']


In [17]:
# ==========================================
# Check Temperature Column
# ==========================================

# 1. View basic statistical summary (min, max, mean, percentiles)
print("=== Temperature Summary Statistics ===")
print(data['body_temperature'].describe())

# 2. Check for missing values
missing_temp = data['body_temperature'].isnull().sum()
print(f"\nMissing temperature values: {missing_temp}")

# 3. Check for physiologically extreme values 
# (Human body temperature normally ranges between ~35°C and ~42°C in clinical settings)
low_temp = data[data['body_temperature'] < 35.0]
high_temp = data[data['body_temperature'] > 42.0]

print(f"\nReadings below 35°C (Hypothermia / Extreme): {len(low_temp)}")
print(f"Readings above 42°C (Hyperthermia / Extreme): {len(high_temp)}")

# 4. Optional: View the actual extreme rows if any exist
if len(low_temp) > 0 or len(high_temp) > 0:
    print("\nExtreme temperature rows:")
    extreme_temps = pd.concat([low_temp, high_temp])
    print(extreme_temps[['inpatient_number', 'body_temperature']])
else:
    print("\nAll temperature values fall within the 35°C – 42°C clinical range.")

=== Temperature Summary Statistics ===
count    2008.000000
mean       36.416484
std         0.439529
min        35.000000
25%        36.200000
50%        36.300000
75%        36.500000
max        42.000000
Name: body_temperature, dtype: float64

Missing temperature values: 0

Readings below 35°C (Hypothermia / Extreme): 0
Readings above 42°C (Hyperthermia / Extreme): 0

All temperature values fall within the 35°C – 42°C clinical range.


In [18]:
data.to_csv('labs_cleaned.csv', index=False)

In [22]:
# ==========================================
# STEP: Analyze Missing Values
# ==========================================
import pandas as pd

# Create a summary dataframe for missing values
missing_summary = pd.DataFrame({
    'Missing Count': data.isnull().sum(),
    'Missing Percentage (%)': (data.isnull().sum() / len(data)) * 100
})

# Filter to show only columns that actually have missing values, sorted from highest to lowest
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values(by='Missing Percentage (%)', ascending=False)

print(f"Total columns with missing values: {len(missing_summary)}")
print("\nTop columns with the highest missing rates:")
print(missing_summary.head(15).to_string())

# Optional: Save the full missing values report to a CSV file
missing_summary.to_csv('missing_values_report.csv')
print("\nFull missing values report saved to 'missing_values_report.csv'.")

Total columns with missing values: 99

Top columns with the highest missing rates:
                                Missing Count  Missing Percentage (%)
cholinesterase                           2008              100.000000
homocysteine                             1862               92.729084
apolipoprotein_a                         1832               91.235060
apolipoprotein_b                         1832               91.235060
lipoprotein                              1832               91.235060
erythrocyte_sedimentation_rate           1701               84.711155
myoglobin                                1610               80.179283
serum_magnesium                          1601               79.731076
inorganic_phosphorus                     1601               79.731076
glutamic_oxaliplatin                     1416               70.517928
high_sensitivity_protein                 1067               53.137450
reduced_hemoglobin                       1016               50.597610
methemo

In [24]:
# 1. Check the data type of every column
print("=== Column Data Types ===")
print(data.dtypes)

# 2. Automatically detect if any object (text) columns contain pure numbers 
# (which tells you if numbers are incorrectly stored as strings)
print("\n=== Checking object columns for numeric values ===")
for col in data.select_dtypes(include=['object']).columns:
    # Try converting a sample to numeric to see if they are actually numbers in disguise
    try:
        sample_converted = pd.to_numeric(df[col].dropna().head(100))
        print(f"Column '{col}' is stored as text (object), but contains numbers.")
    except ValueError:
        pass

=== Column Data Types ===
inpatient_number             int64
body_temperature           float64
pulse                        int64
respiration                  int64
systolic_blood_pressure      int64
                            ...   
partial_oxygen_pressure    float64
oxyhemoglobin              float64
anion_gap                  float64
free_calcium               float64
total_hemoglobin           float64
Length: 107, dtype: object

=== Checking object columns for numeric values ===


In [26]:
# ==========================================
# INSPECT: Check for Data Type Mismatches
# ==========================================

mismatch_report = []

for col in data.columns:
    current_type = data[col].dtype
    series_clean = data[col].dropna()
    
    if len(series_clean) == 0:
        continue
        
    # If the column is stored as text (object), check if it contains numbers
    if current_type == 'object':
        numeric_conversion = pd.to_numeric(series_clean, errors='coerce')
        num_success = numeric_conversion.notnull().sum()
        
        if num_success > 0 and num_success < len(series_clean):
            mismatch_report.append({
                'Column': col,
                'Stored As': 'object (Text)',
                'Mismatch Type': 'Mixed content (contains both text strings and numbers)'
            })
        elif num_success == len(series_clean):
            mismatch_report.append({
                'Column': col,
                'Stored As': 'object (Text)',
                'Mismatch Type': 'Entirely numeric values stored as strings'
            })

# Display the findings safely
if mismatch_report:
    mismatch_df = pd.DataFrame(mismatch_report)
    print("=== Potential Data Type Mismatches Found ===")
    print(mismatch_df.to_string(index=False))
else:
    print("=== No data type mismatches detected. All text/numeric types look consistent. ===")

=== No data type mismatches detected. All text/numeric types look consistent. ===


In [28]:
# Select only numerical columns, but exclude ID or number columns
numeric_cols = data.select_dtypes(include=[np.number]).columns
cols_to_check = [col for col in numeric_cols if 'number' not in col.lower() and 'id' not in col.lower()]

# Generate summary, round decimals, and transpose
summary_clean = data[cols_to_check].agg(['min', 'max', 'mean']).T.round(2)

# Display as a clean, structured table in Jupyter
display(summary_clean)

,min,max,mean
body_temperature,35.00,42.00,36.42
pulse,0.00,198.00,85.24
respiration,0.00,36.00,19.09
systolic_blood_pressure,0.00,252.00,131.06
diastolic_blood_pressure,0.00,146.00,76.57
...,...,...,...
partial_oxygen_pressure,20.00,255.00,108.12
oxyhemoglobin,24.30,99.10,94.94
anion_gap,-1.20,43.70,14.02
free_calcium,0.89,1.39,1.11


In [29]:
# ==========================================
# INSPECT: Check Percentage of Zeros in Columns
# ==========================================

# Select numerical columns, excluding ID or number columns
numeric_cols = data.select_dtypes(include=[np.number]).columns
cols_to_check = [col for col in numeric_cols if 'number' not in col.lower() and 'id' not in col.lower()]

# Calculate zero counts and percentages
zero_stats = []
for col in cols_to_check:
    # Count how many exact 0s are in the column (ignoring NaN)
    zero_count = (data[col] == 0).sum()
    total_valid = data[col].notnull().sum()
    
    if zero_count > 0:
        zero_pct = (zero_count / total_valid) * 100
        zero_stats.append({
            'Column': col, 
            'Zero Count': zero_count, 
            'Total Valid Rows': total_valid,
            'Zero Percentage (%)': round(zero_pct, 2)
        })

# Convert to a clean DataFrame and display as an HTML table
zero_report_df = pd.DataFrame(zero_stats)

if not zero_report_df.empty:
    display(zero_report_df.sort_values(by='Zero Percentage (%)', ascending=False))
else:
    print("No columns contain a value of 0.")

,Column,Zero Count,Total Valid Rows,Zero Percentage (%)
8,eosinophil_count,202,1981,10.20
7,eosinophil_ratio,198,1981,9.99
11,methemoglobin,87,992,8.77
12,carboxyhemoglobin,36,992,3.63
6,basophil_count,54,1981,2.73
10,high_sensitivity_troponin,51,1929,2.64
5,basophil_ratio,29,1981,1.46
9,d_dimer,4,1840,0.22
2,systolic_blood_pressure,3,2008,0.15
3,diastolic_blood_pressure,3,2008,0.15


In [31]:
data = data.rename(columns={'inpatient_number': 'Patient_Id'})

print("Column renamed successfully!")
print("Updated column names snippet:", data.columns[:5].tolist())

Column renamed successfully!
Updated column names snippet: ['Patient Id', 'body_temperature', 'pulse', 'respiration', 'systolic_blood_pressure']


In [32]:
print(list(data.columns))

['Patient Id', 'body_temperature', 'pulse', 'respiration', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value', 'fio2', 'creatinine_enzymatic_method', 'urea', 'uric_acid', 'glomerular_filtration_rate', 'cystatin', 'white_blood_cell', 'monocyte_ratio', 'monocyte_count', 'red_blood_cell', 'coefficient_of_variation_of_red_blood_cell_distribution_width', 'standard_deviation_of_red_blood_cell_distribution_width', 'mean_corpuscular_volume', 'hematocrit', 'lymphocyte_count', 'mean_hemoglobin_volume', 'mean_hemoglobin_concentration', 'mean_platelet_volume', 'basophil_ratio', 'basophil_count', 'eosinophil_ratio', 'eosinophil_count', 'hemoglobin', 'platelet', 'platelet_distribution_width', 'platelet_hematocrit', 'neutrophil_ratio', 'neutrophil_count', 'd_dimer', 'international_normalized_ratio', 'activated_partial_thromboplastin_time', 'thrombin_time', 'prothrombin_activity', 'prothrombin_time_ratio', 'fibrinogen', 'high_sensitivity_troponin', 'myoglobin', 'carbon_dioxide_binding_

In [33]:
data = data.rename(columns={'Patient Id': 'Patient_Id'})

print("Column renamed successfully!")
print("Updated column names snippet:", data.columns[:5].tolist())

Column renamed successfully!
Updated column names snippet: ['Patient_Id', 'body_temperature', 'pulse', 'respiration', 'systolic_blood_pressure']


In [34]:
print(list(data.columns))

['Patient_Id', 'body_temperature', 'pulse', 'respiration', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value', 'fio2', 'creatinine_enzymatic_method', 'urea', 'uric_acid', 'glomerular_filtration_rate', 'cystatin', 'white_blood_cell', 'monocyte_ratio', 'monocyte_count', 'red_blood_cell', 'coefficient_of_variation_of_red_blood_cell_distribution_width', 'standard_deviation_of_red_blood_cell_distribution_width', 'mean_corpuscular_volume', 'hematocrit', 'lymphocyte_count', 'mean_hemoglobin_volume', 'mean_hemoglobin_concentration', 'mean_platelet_volume', 'basophil_ratio', 'basophil_count', 'eosinophil_ratio', 'eosinophil_count', 'hemoglobin', 'platelet', 'platelet_distribution_width', 'platelet_hematocrit', 'neutrophil_ratio', 'neutrophil_count', 'd_dimer', 'international_normalized_ratio', 'activated_partial_thromboplastin_time', 'thrombin_time', 'prothrombin_activity', 'prothrombin_time_ratio', 'fibrinogen', 'high_sensitivity_troponin', 'myoglobin', 'carbon_dioxide_binding_

In [35]:
# ==========================================
# INSPECT: Check Systolic vs. Diastolic Logic
# ==========================================


systolic_col = 'systolic_blood_pressure'
diastolic_col = 'diastolic_blood_pressure'

# Find rows where systolic is NOT strictly greater than diastolic
# (We also make sure to ignore rows where either value is missing/NaN)
invalid_bp = data[
    data[systolic_col].notnull() & 
    data[diastolic_col].notnull() & 
    (data[systolic_col] <= data[diastolic_col])
]

print(f"Total rows checked: {data[systolic_col].notnull().sum()}")
print(f"Rows where Systolic is NOT higher than Diastolic: {len(invalid_bp)}")

# If any invalid rows exist, display a preview of them
if len(invalid_bp) > 0:
    print("\nPreview of rows with logic contradictions:")
    display(invalid_bp[['patient_id', systolic_col, diastolic_col]].head(10))
else:
    print("\nAll records pass! Systolic is consistently higher than Diastolic.")

Total rows checked: 2008
Rows where Systolic is NOT higher than Diastolic: 5

Preview of rows with logic contradictions:


KeyError: "['patient_id'] not in index"

In [36]:
# Automatically find the ID column name safely
id_col = [col for col in data.columns if 'patient' in col.lower() or 'id' in col.lower()][0]

if len(invalid_bp) > 0:
    print("\nPreview of rows with logic contradictions:")
    display(invalid_bp[[id_col, systolic_col, diastolic_col]].head(10))
else:
    print("\nAll records pass! Systolic is consistently higher than Diastolic.")


Preview of rows with logic contradictions:


,Patient_Id,systolic_blood_pressure,diastolic_blood_pressure
533,754892,0,0
611,764993,0,0
691,825901,73,95
744,838870,117,118
1831,773886,0,0
